In [4]:
import numpy as np
import pandas as pd

# 1. Membuat data historis tiruan sebanyak 1000 baris
np.random.seed(42)
jumlah_data = 1000

# Input: 0=Senin, 1=Selasa, 2=Rabu, 3=Kamis, 4=Jumat, 5=Sabtu
data_hari = np.random.choice(6, size=jumlah_data)

# 2. Membuat 10 kolom slot parkir (0 = Kosong, 1 = Terisi) berdasarkan pola hari
layout_list = []
for hari in data_hari:
    if hari in [0, 4]:  # Senin & Jumat ramai (peluang kosong kecil)
        prob_kosong = 0.2
    elif hari in [2, 5]:  # Rabu & Sabtu sepi (peluang kosong besar)
        prob_kosong = 0.7
    else:  # Selasa & Kamis normal
        prob_kosong = 0.4

    # Bangkitkan 10 slot sekaligus untuk baris ini
    slots = np.random.choice([0, 1], size=10, p=[prob_kosong, 1 - prob_kosong])
    layout_list.append(slots)

# 3. Gabungkan menjadi satu DataFrame yang rapi
df = pd.DataFrame({"hari": data_hari})

# Membuat nama kolom Slot_1 sampai Slot_10 secara otomatis
nama_kolom_slot = [f"Slot_{i+1}" for i in range(10)]
df_slots = pd.DataFrame(layout_list, columns=nama_kolom_slot)

# Gabungkan kolom hari dan 10 kolom slot parkir
dataset_parkir = pd.concat([df, df_slots], axis=1)

print(f"Dataset berhasil dibuat dengan total: {dataset_parkir.shape[0]} baris.")
print("\nContoh 5 baris pertama data Anda:")
print(dataset_parkir.head())


Dataset berhasil dibuat dengan total: 1000 baris.

Contoh 5 baris pertama data Anda:
   hari  Slot_1  Slot_2  Slot_3  Slot_4  Slot_5  Slot_6  Slot_7  Slot_8  \
0     3       0       0       0       0       0       0       0       1   
1     4       1       1       0       1       1       1       1       1   
2     2       1       0       0       0       1       0       0       0   
3     4       0       1       1       1       0       1       1       0   
4     4       0       1       1       1       1       1       1       1   

   Slot_9  Slot_10  
0       0        1  
1       0        1  
2       0        0  
3       1        0  
4       1        1  


In [ ]:
import pickle
from sklearn.tree import DecisionTreeClassifier

# Memisahkan Fitur/Input (X) dan Target/Output (y)
X = dataset_parkir[["hari"]]
y = dataset_parkir[nama_kolom_slot]

# Latih model Decision Tree untuk memprediksi ke-10 kolom slot secara bersamaan
model = DecisionTreeClassifier(random_state=42)
model.fit(X, y)

# Hitung hari apa yang diprediksi paling banyak menghasilkan nilai '0' (kosong)
nama_hari = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu"]
total_kosong_per_hari = {}

for i in range(6):
    # Buat DataFrame input sementara agar sesuai dengan nama fitur saat training
    X_test = pd.DataFrame({"hari": [i]})
    prediksi_layout = model.predict(X_test)

    # Hitung berapa banyak angka 0 di seluruh 10 slot tersebut
    jumlah_kosong = np.sum(prediksi_layout == 0)
    total_kosong_per_hari[nama_hari[i]] = int(jumlah_kosong)

hari_terbanyak = max(total_kosong_per_hari, key=total_kosong_per_hari.get)
print(f"✅ Model ML Selesai Dilatih!")
print(
    f"💡 Prediksi Tren: Hari paling banyak KOSONG adalah {hari_terbanyak} ({total_kosong_per_hari[hari_terbanyak]} slot)."
)

# Ekspor model yang sudah dilatih menjadi file biner
with open("model_parkir.pkl", "wb") as file:
    pickle.dump(model, file)
print("📦 Model diekspor dengan sukses sebagai 'model_parkir.pkl'")


Model ML Selesai Dilatih!
Prediksi Tren: Hari paling banyak KOSONG adalah Rabu (10 slot).
Model diekspor dengan sukses sebagai 'model_parkir.pkl'
